# Day 1 — Tokenization & Next-Token Prediction

How LLMs break text into tokens, convert them to embeddings, and predict the next token one step at a time. This is the foundational mechanism behind every LLM response, regardless of model size.

## Setup
Install the required libraries (only needed once per Colab session).

In [ ]:
!pip install transformers torch --quiet

## Load the model and tokenizer
We're using GPT-2 — small, fast, and enough to demonstrate the core mechanism without needing a large download.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

## Tokenize a sentence
This is the first step in our pipeline: text → tokens. Note that GPT-2 uses byte-level BPE, so some words split into sub-pieces, and a leading space shows up as `Ġ` in the raw token view (not a bug — just how spacing is encoded).

In [ ]:
text = "My firewall keeps blocking"
inputs = tokenizer(text, return_tensors="pt")

print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

## Run the model and inspect its prediction
`torch.no_grad()` disables gradient tracking since we're only doing inference (not training), which saves memory and speeds things up.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

next_token_logits = outputs.logits[0, -1]
top5 = torch.topk(next_token_logits, 5)

print("Top 5 predicted next tokens:")
for score, idx in zip(top5.values, top5.indices):
    print(f"  {tokenizer.decode([idx]):15s} score: {score.item():.2f}")

## What to notice
- Scores are logits, not probabilities — negative numbers are normal, only the ranking matters.
- GPT-2 has no domain-specific knowledge, so for a cybersecurity-flavored sentence it defaults to generic completions (the, all, access) rather than anything security-specific.
- This gap — a general model not knowing your domain — is exactly what fine-tuning and RAG (covered in later chapters) are built to close.

**Next chapter:** transformer architecture and attention — how the model decides which earlier words matter most when predicting the next one.